In [1]:
# K-1 (Sel 1) — Instal pustaka & mount Google Drive
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# K-1 (Sel 2) — Path folder + ekstrak trips_bersih.zip dari Praktikum 3

import os, glob, zipfile

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
os.makedirs(DIR_SIMPAN, exist_ok=True)

PAKAI_DATA_TIRUAN = False   # tetap False: kita pakai data asli dari trips_bersih.zip

ZIP_P3 = f"{DIR_P3}/trips_bersih.zip"
TUJUAN_EKSTRAK = "/content/trips_bersih_p3"     # diekstrak ke disk Colab (lebih cepat dibaca)

assert os.path.exists(ZIP_P3), f"File {ZIP_P3} tidak ada"
if not os.path.exists(TUJUAN_EKSTRAK):
    with zipfile.ZipFile(ZIP_P3) as z:
        z.extractall(TUJUAN_EKSTRAK)
    print("Zip diekstrak ke", TUJUAN_EKSTRAK)

# Cari folder yang berisi subfolder partisi "borough_naik=..." (struktur isi zip bisa berbeda-beda)
partisi = glob.glob(f"{TUJUAN_EKSTRAK}/**/borough_naik=*", recursive=True)
if partisi:
    TRIPS_BERSIH = os.path.dirname(partisi[0])
else:
    # zip tanpa partisi: pakai folder yang berisi file .parquet
    file_pq = glob.glob(f"{TUJUAN_EKSTRAK}/**/*.parquet", recursive=True)
    assert file_pq, "Tidak ada file .parquet di dalam trips_bersih.zip"
    TRIPS_BERSIH = os.path.dirname(file_pq[0])

print("TRIPS_BERSIH =", TRIPS_BERSIH)
print("Isi folder:", sorted(os.listdir(TRIPS_BERSIH)))
print("Jumlah file parquet:", len(glob.glob(f"{TRIPS_BERSIH}/**/*.parquet", recursive=True)))

Zip diekstrak ke /content/trips_bersih_p3
TRIPS_BERSIH = /content/trips_bersih_p3
Isi folder: ['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens', 'borough_naik=Staten%20Island', 'borough_naik=Unknown', 'borough_naik=__HIVE_DEFAULT_PARTITION__']
Jumlah file parquet: 16


In [5]:
# K-1b — Jalur cadangan: data tiruan (HANYA jika PAKAI_DATA_TIRUAN = True)

import numpy as np, pandas as pd, os, shutil

def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})

    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)

    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])

    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

if PAKAI_DATA_TIRUAN:
    TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")
    print("PERHATIAN: notebook ini MEMAKAI DATA TIRUAN — wajib ditulis di laporan.")
else:
    print("K-1b dilewati: memakai trips_bersih/ asli dari Praktikum 3.")

K-1b dilewati: memakai trips_bersih/ asli dari Praktikum 3.


In [7]:
# K-1 (Sel 3) — Import, fungsi ukur(), baca data, ganti nama kolom
import os, time, json, sqlite3, random, shutil, glob
from urllib.parse import unquote
import numpy as np, pandas as pd

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

def baca_trips(folder):
    bagian = []
    for f in sorted(glob.glob(f"{folder}/**/*.parquet", recursive=True)):
        t = pd.read_parquet(f)
        # kolom kategori -> teks biasa, supaya semua bagian bisa digabung
        for c in t.select_dtypes("category").columns:
            t[c] = t[c].astype(object)
        # ambil nilai partisi dari nama folder, mis. "borough_naik=Staten%20Island"
        nama_dir = os.path.basename(os.path.dirname(f))
        if "=" in nama_dir:
            kolom, nilai = nama_dir.split("=", 1)
            if kolom not in t.columns:
                nilai = unquote(nilai)
                t[kolom] = None if nilai == "__HIVE_DEFAULT_PARTITION__" else nilai
        bagian.append(t)
    return pd.concat(bagian, ignore_index=True)

df = ukur("baca trips_bersih", lambda: baca_trips(TRIPS_BERSIH))

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])
print("wilayah:", df["wilayah_jemput"].value_counts().to_dict())

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

[baca trips_bersih] 3.4053 s
baris: 5,973,820 | kolom: 18
wilayah: {'Manhattan': 5319218, 'Queens': 540630, 'Unknown': 75218, 'Brooklyn': 31448, 'Bronx': 6148, 'None': 680, 'Staten Island': 422, 'EWR': 56}


In [8]:
# K-2 — Tabel biasa (SQLite) vs key-value (dict Python)
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relational: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relational: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")    # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)

[relational: muat tabel] 1.0382 s
[key-value: bangun dict] 4.7989 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 4.6329 s
[cari 200 perjalanan - SQL dengan indeks] 0.0056 s
[cari 200 perjalanan - key-value] 0.0019 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.3624 s
[hitung per wilayah - key-value (buka semua nilai)] 2.3904 s
[('Manhattan', 266924, 22.7), ('Queens', 27324, 70.41), ('Unknown', 3767, 38.57)]
hasil sama? True


In [9]:
# K-3 (Sel 1) — Simpan perjalanan sebagai dokumen JSON bersarang (TinyDB)
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya":  {"jarak_mil": float(r.jarak_mil),
                   "total": float(r.total_bayar),
                   "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-29 09:30:32",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 246
  },
  "biaya": {
    "jarak_mil": 0.88,
    "total": 14.28,
    "metode_bayar": "Kartu kredit",
    "tip": 2.38
  }
}


In [10]:
# K-3 (Sel 2) — Query dokumen + menambah field baru tanpa ALTER TABLE
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") & (Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip": 3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 662
dokumen yang punya field tip     : 15957
dokumen yang TIDAK punya tip     : 4043
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


In [11]:
# K-4 (Sel 1) — DuckDB langsung pada Parquet vs SQLite per baris
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
if not glob.glob(GLOB):                       # jaga-jaga bila struktur folder berbeda
    GLOB = f"{TRIPS_BERSIH}/**/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID         AS zona_jemput,
           DOLocationID         AS zona_tujuan,
           borough_naik         AS wilayah_jemput,
           trip_distance        AS jarak_mil,
           total_amount         AS total_bayar,
           nama_pembayaran      AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda: con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s, index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in hasil_s])

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.2283 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  5319218       22.72
1                      Queens   540630       70.46
2                     Unknown    75218       39.24
3                    Brooklyn    31448       32.72
4                       Bronx     6148       34.44
5  __HIVE_DEFAULT_PARTITION__      680       74.72
6               Staten Island      422       74.07
7                         EWR       56      106.41
[SQLite - muat seluruh baris ke tabel] 16.8721 s
[SQLite - hitung per wilayah] 6.4580 s
hasil sama? False


In [12]:
# K-4 (Sel 2) — Partition pruning + ukuran file
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))

ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0315 s
[(422, 74.07)]
Parquet (18 kolom): 133.1 MB | SQLite (5 kolom): 230.4 MB


In [13]:
# K-5 (Sel 1) — Bangun graph arus antar zona
import networkx as nx

arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.4361 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 44130), (236, 237, 37620), (236, 236, 28220), (237, 237, 27506), (264, 264, 27124)]
SQL  : [(237, 236, 44130), (236, 237, 37620), (236, 236, 28220), (237, 237, 27506), (264, 264, 27124)]
zona tujuan terpopuler: [(236, 288418), (237, 260738), (161, 227982), (230, 175412), (170, 174220)]


In [14]:
# K-5 (Sel 2) — Zona terjangkau dalam 2 langkah: graph vs SQL self-join
MIN = 50                       # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0]      # mulai dari zona tujuan terpopuler

def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?", hg == hs)

H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0080 s
[2 langkah - SQL self-join (DuckDB)] 0.0161 s
zona awal: 236 | terjangkau (graph): 226 | (SQL): 226 | sama? True
dalam <= 1 langkah: 91 zona
dalam <= 2 langkah: 226 zona
dalam <= 3 langkah: 226 zona
dalam <= 4 langkah: 226 zona


In [15]:
# K-6 — Loop Python vs NumPy vs indeks STRtree
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)     # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)      # latitude
titik = points(bujur, lintang)

M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y + 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?", h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 0.7051 s
[cara 2 - NumPy, 1000 kotak] 0.5892 s
[bangun indeks STRtree] 0.1853 s
[cara 3 - STRtree, 1000 kotak] 0.3000 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


In [17]:
# K-7 (Sel 1) — Membuat tabel Delta: tulis, tambah, koreksi
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 27
versi: 2


In [18]:
# K-7 (Sel 2) — Time travel, riwayat, log transaksi, penegakan skema
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:", n_koreksi)

for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 27.0 | jumlah baris dikoreksi: 27
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


In [19]:
# K-8 — 20 append kecil, lalu compaction, lalu VACUUM (dry run)
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA, potongan, mode="append")
dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(),
      "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter() - t0

hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter() - t0
print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(),
      "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)   # dry_run = hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0767 s | sesudah: 0.0779 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


In [20]:
# Fungsi bantu latihan: ukur beberapa kali, ambil median (lebih stabil)
def ukur_median(label, fungsi, ulang=5):
    fungsi()                                   # pemanasan (cache), tidak dihitung
    waktu = []
    for _ in range(ulang):
        t0 = time.perf_counter()
        hasil = fungsi()
        waktu.append(time.perf_counter() - t0)
    detik = float(np.median(waktu))
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s (median {ulang}x)")
    return detik, hasil

In [21]:
# LATIHAN 1 — Pencarian 2.000 id: tabel berindeks vs key-value (µs per pencarian)
random.seed(2024)
id_uji_2000 = random.sample(range(1, len(data) + 1), 2000)

def cari_sql_2000():
    cur = con.cursor()          # tabel ini sudah punya indeks idx_id dari K-2
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_uji_2000]

def cari_kv_2000():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]

t_sql_2000, hasil_sql = ukur_median("Latihan 1 - cari 2.000 id - SQL dengan indeks", cari_sql_2000)
t_kv_2000, hasil_kv = ukur_median("Latihan 1 - cari 2.000 id - key-value", cari_kv_2000)

us_sql = t_sql_2000 / len(id_uji_2000) * 1e6
us_kv = t_kv_2000 / len(id_uji_2000) * 1e6
print(f"rata-rata per pencarian -> SQL berindeks: {us_sql:.2f} µs | key-value: {us_kv:.2f} µs")
print(f"key-value {us_sql / us_kv:.1f}x lebih cepat per pencarian")
print("hasil sama?", [r[0] for r in hasil_sql] == [d["id_perjalanan"] for d in hasil_kv])

[Latihan 1 - cari 2.000 id - SQL dengan indeks] 0.0334 s (median 5x)
[Latihan 1 - cari 2.000 id - key-value] 0.0110 s (median 5x)
rata-rata per pencarian -> SQL berindeks: 16.70 µs | key-value: 5.49 µs
key-value 3.0x lebih cepat per pencarian
hasil sama? True


In [23]:
# LATIHAN 2 — Hitung dokumen zona tujuan X + "Tunai": TinyDB vs SQL (tabel K-2)

from collections import Counter

METODE = "Tunai"
print("metode_bayar yang ada di dokumen:", Counter(d["biaya"]["metode_bayar"] for d in tabel.all()).most_common())

ZONA_TUJUAN = Counter(d["tujuan"]["zona"] for d in tabel.all()
                      if d["biaya"]["metode_bayar"] == METODE).most_common(1)[0][0]
print("zona tujuan yang diuji:", ZONA_TUJUAN)

Q = Query()
n_dokumen = len(tabel.search((Q.tujuan.zona == ZONA_TUJUAN) & (Q.biaya.metode_bayar == METODE)))

n_sql_semua = con.execute(
    "SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = ?",
    (ZONA_TUJUAN, METODE)).fetchone()[0]

n_sql_sama = con.execute(
    "SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan = ? AND metode_bayar = ? AND id_perjalanan <= 20000",
    (ZONA_TUJUAN, METODE)).fetchone()[0]

print(f"TinyDB (20.000 dokumen)         : {n_dokumen}")
print(f"SQL tabel K-2 (300.000 baris)   : {n_sql_semua}")
print(f"SQL dibatasi id 1..20.000       : {n_sql_sama}")
print("TinyDB vs SQL penuh sama?", n_dokumen == n_sql_semua, "| TinyDB vs SQL id 1..20.000 sama?", n_dokumen == n_sql_sama)

metode_bayar yang ada di dokumen: [('Kartu kredit', 15958), ('Tunai', 3432), (None, 432), ('Sengketa', 120), ('Gratis', 59)]
zona tujuan yang diuji: 237
TinyDB (20.000 dokumen)         : 151
SQL tabel K-2 (300.000 baris)   : 2206
SQL dibatasi id 1..20.000       : 151
TinyDB vs SQL penuh sama? False | TinyDB vs SQL id 1..20.000 sama? True


In [24]:
# LATIHAN 3 — DuckDB: SUM 1 kolom vs SUM semua kolom angka (column pruning)
FROM_PQ = f"read_parquet('{GLOB}', hive_partitioning=true)"

skema = con_d.sql(f"DESCRIBE SELECT * FROM {FROM_PQ}").df()
pola_angka = r"(BIGINT|INTEGER|SMALLINT|TINYINT|HUGEINT|UBIGINT|UINTEGER|DOUBLE|FLOAT|REAL|DECIMAL)"
kolom_angka = skema.loc[skema["column_type"].str.match(pola_angka), "column_name"].tolist()
print(f"{len(kolom_angka)} kolom angka:", kolom_angka)

sql_satu = f"SELECT SUM(total_amount) FROM {FROM_PQ}"
sql_banyak = "SELECT " + ", ".join(f'SUM("{c}")' for c in kolom_angka) + f" FROM {FROM_PQ}"

t_satu, _ = ukur_median("Latihan 3 - SUM 1 kolom (total_amount)", lambda: con_d.sql(sql_satu).fetchall())
t_banyak, _ = ukur_median(f"Latihan 3 - SUM {len(kolom_angka)} kolom sekaligus", lambda: con_d.sql(sql_banyak).fetchall())
print(f"SUM {len(kolom_angka)} kolom {t_banyak / t_satu:.1f}x lebih lama daripada SUM 1 kolom")

# Bukti column pruning: berapa MB data (terkompresi) yang harus dibaca
try:
    meta = con_d.sql(f"""SELECT path_in_schema AS kolom, SUM(total_compressed_size) / 1024 / 1024 AS mb
                         FROM parquet_metadata('{GLOB}') GROUP BY 1""").df()
    mb_satu = meta.loc[meta["kolom"] == "total_amount", "mb"].sum()
    mb_banyak = meta.loc[meta["kolom"].isin(kolom_angka), "mb"].sum()
    print(f"data yang dibaca -> 1 kolom: {mb_satu:.1f} MB | {len(kolom_angka)} kolom: {mb_banyak:.1f} MB")
except Exception as e:
    print("parquet_metadata tidak tersedia:", e)

12 kolom angka: ['passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'tip_amount', 'total_amount', 'durasi_menit', 'jam', 'suhu_c', 'hujan_mm']
[Latihan 3 - SUM 1 kolom (total_amount)] 0.1305 s (median 5x)
[Latihan 3 - SUM 12 kolom sekaligus] 1.2731 s (median 5x)
SUM 12 kolom 9.8x lebih lama daripada SUM 1 kolom
data yang dibaca -> 1 kolom: 10.8 MB | 12 kolom: 63.5 MB


In [25]:
# LATIHAN 4 — 5 zona dengan tujuan berbeda terbanyak: out-degree vs COUNT(DISTINCT)
top_out_graf = sorted(G.out_degree(), key=lambda kv: (-kv[1], kv[0]))[:5]

top_out_sql = con_d.sql("""
    SELECT zona_jemput, COUNT(DISTINCT zona_tujuan) AS jumlah_tujuan
    FROM perjalanan GROUP BY zona_jemput
    ORDER BY jumlah_tujuan DESC, zona_jemput LIMIT 5""").fetchall()

print("graph:", [(int(a), int(b)) for a, b in top_out_graf])
print("SQL  :", [(int(a), int(b)) for a, b in top_out_sql])
print("sama?", [(int(a), int(b)) for a, b in top_out_graf] == [(int(a), int(b)) for a, b in top_out_sql])

graph: [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL  : [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
sama? True


In [26]:
# LATIHAN 5 — Ukuran kotak 0,02 vs 0,01 (K-6) vs 0,001 derajat: STRtree vs NumPy
def uji_ukuran_kotak(sisi):
    h = sisi / 2
    kotak_s = [box(x - h, y - h, x + h, y + h) for x, y in zip(cx, cy)]

    def pakai_numpy():
        return [int(((bujur >= x - h) & (bujur <= x + h) & (lintang >= y - h) & (lintang <= y + h)).sum())
                for x, y in zip(cx, cy)]

    def pakai_strtree():
        return [len(pohon.query(k, predicate="contains")) for k in kotak_s]

    t_np, hasil_np = ukur_median(f"Latihan 5 - NumPy, sisi {sisi} derajat", pakai_numpy, ulang=3)
    t_idx, hasil_idx = ukur_median(f"Latihan 5 - STRtree, sisi {sisi} derajat", pakai_strtree, ulang=3)
    return {"sisi (derajat)": sisi, "titik ditemukan": sum(hasil_idx),
            "NumPy (s)": round(t_np, 4), "STRtree (s)": round(t_idx, 4),
            "STRtree lebih cepat (x)": round(t_np / t_idx, 1), "hasil sama?": hasil_np == hasil_idx}

hasil_l5 = pd.DataFrame([uji_ukuran_kotak(s) for s in (0.02, 0.01, 0.001)])
print(hasil_l5.to_string(index=False))

[Latihan 5 - NumPy, sisi 0.02 derajat] 0.6045 s (median 3x)
[Latihan 5 - STRtree, sisi 0.02 derajat] 1.0258 s (median 3x)
[Latihan 5 - NumPy, sisi 0.01 derajat] 0.4840 s (median 3x)
[Latihan 5 - STRtree, sisi 0.01 derajat] 0.2103 s (median 3x)
[Latihan 5 - NumPy, sisi 0.001 derajat] 0.5249 s (median 3x)
[Latihan 5 - STRtree, sisi 0.001 derajat] 0.0109 s (median 3x)
 sisi (derajat)  titik ditemukan  NumPy (s)  STRtree (s)  STRtree lebih cepat (x)  hasil sama?
          0.020          1666631     0.6045       1.0258                      0.6         True
          0.010           417545     0.4840       0.2103                      2.3         True
          0.001             4149     0.5249       0.0109                     48.1         True


In [27]:
# LATIHAN 6 — Time travel vs restore
dt = DeltaTable(PATH_DELTA)
versi_sebelum = dt.version()
print("versi sebelum restore:", versi_sebelum, "| baris:", len(dt.to_pandas()))
dt.restore(1)                       # kembalikan tabel ke keadaan versi 1 (dicatat sebagai versi baru)
dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

# Bukti riwayat TIDAK terhapus: versi sebelum restore masih bisa dibaca
print(f"time travel ke versi {versi_sebelum} (sebelum restore) -> baris:",
      len(DeltaTable(PATH_DELTA, version=versi_sebelum).to_pandas()))

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']
time travel ke versi 23 (sebelum restore) -> baris: 160000


K-9. Merangkum

In [28]:
# K-9 (Sel 1) — Simpan semua angka waktu ke benchmark_nosql.csv
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih  3.4053
                             relational: muat tabel  1.0382
                             key-value: bangun dict  4.7989
             cari 200 perjalanan - SQL tanpa indeks  4.6329
            cari 200 perjalanan - SQL dengan indeks  0.0056
                    cari 200 perjalanan - key-value  0.0019
                  hitung per wilayah - SQL GROUP BY  0.3624
  hitung per wilayah - key-value (buka semua nilai)  2.3904
DuckDB - hitung per wilayah (langsung dari Parquet)  0.2283
               SQLite - muat seluruh baris ke tabel 16.8721
                        SQLite - hitung per wilayah  6.4580
               DuckDB - hanya wilayah Staten Island  0.0315
                             hitung arus antar zona  0.4361
                       2 langkah - graph (NetworkX)  0.0080
                 2 langkah - SQL self-join (DuckDB)  0.0161
                      cara 1 - loop Pyth

In [29]:
# K-9 (Sel 2) — Tabel rekap, kolom "Angka Anda" terisi otomatis
def w(label):
    nilai = [c["detik"] for c in catatan if c["langkah"] == label]
    return max(nilai[-1], 0.0001) if nilai else float("nan")

a = w("cari 200 perjalanan - SQL tanpa indeks")
b = w("cari 200 perjalanan - SQL dengan indeks")
c = w("cari 200 perjalanan - key-value")
d_ = w("DuckDB - hitung per wilayah (langsung dari Parquet)")
e = w("SQLite - hitung per wilayah")
f_ = w("hitung per wilayah - key-value (buka semua nilai)")
g = w("hitung per wilayah - SQL GROUP BY")
h_g = w("2 langkah - graph (NetworkX)")
h_s = w("2 langkah - SQL self-join (DuckDB)")
s_loop = w("cara 1 - loop Python, 5 kotak") * 200
s_np = w("cara 2 - NumPy, 1000 kotak")
s_idx = w("bangun indeks STRtree") + w("cara 3 - STRtree, 1000 kotak")

tabel_rekap = pd.DataFrame([
    ["Ambil satu data lewat id", "Key-value; tabel dengan indeks", "Tabel tanpa indeks",
     f"200 id: tanpa indeks {a:.4f} s, dengan indeks {b:.4f} s ({a/b:.0f}x), key-value {c:.4f} s"],
    ["Menghitung atau merangkum banyak baris", "Per kolom (DuckDB pada Parquet)", "Key-value (buka semua nilai)",
     f"DuckDB {d_:.4f} s vs SQLite {e:.4f} s ({e/d_:.1f}x); key-value {f_:.4f} s vs SQL {g:.4f} s ({f_/g:.1f}x lebih lambat)"],
    ["Data dengan field opsional atau berubah-ubah", "Document", "Tabel (kolom NULL, migrasi)", "(kualitatif)"],
    ["Keterhubungan dengan jumlah langkah bebas", "Graph", "SQL (satu JOIN per langkah)",
     f"2 langkah: graph {h_g:.4f} s vs SQL {h_s:.4f} s; hasil sama: {hg == hs}"],
    ["Mencari lokasi dalam wilayah", "Spasial dengan indeks", "Memeriksa semua titik",
     f"1.000 kotak: loop ≈ {s_loop:.0f} s (perkiraan), NumPy {s_np:.4f} s, STRtree {s_idx:.4f} s ({s_np/s_idx:.1f}x vs NumPy)"],
    ["Perubahan yang harus aman dan bisa dibatalkan", "Tabel Delta", "Folder Parquet biasa", "(kualitatif)"],
], columns=["Kebutuhan", "Cara yang unggul", "Cara yang lemah", "Angka Anda"])

pd.set_option("display.max_colwidth", None)
display(tabel_rekap)
tabel_rekap.to_csv(f"{DIR_SIMPAN}/tabel_rekap_k9.csv", index=False)

,Kebutuhan,Cara yang unggul,Cara yang lemah,Angka Anda
0,Ambil satu data lewat id,Key-value; tabel dengan indeks,Tabel tanpa indeks,"200 id: tanpa indeks 4.6329 s, dengan indeks 0.0056 s (827x), key-value 0.0019 s"
1,Menghitung atau merangkum banyak baris,Per kolom (DuckDB pada Parquet),Key-value (buka semua nilai),DuckDB 0.2283 s vs SQLite 6.4580 s (28.3x); key-value 2.3904 s vs SQL 0.3624 s (6.6x lebih lambat)
2,Data dengan field opsional atau berubah-ubah,Document,"Tabel (kolom NULL, migrasi)",(kualitatif)
3,Keterhubungan dengan jumlah langkah bebas,Graph,SQL (satu JOIN per langkah),2 langkah: graph 0.0080 s vs SQL 0.0161 s; hasil sama: True
4,Mencari lokasi dalam wilayah,Spasial dengan indeks,Memeriksa semua titik,"1.000 kotak: loop ≈ 141 s (perkiraan), NumPy 0.5892 s, STRtree 0.4853 s (1.2x vs NumPy)"
5,Perubahan yang harus aman dan bisa dibatalkan,Tabel Delta,Folder Parquet biasa,(kualitatif)


In [31]:
# TUGAS 3 (Sel 1) — Simulasi dua hari kerja + baca keadaan akhir hari 1
from datetime import datetime, timezone

PATH_2HARI = "/content/trips_delta_2hari"
if os.path.exists(PATH_2HARI):
    shutil.rmtree(PATH_2HARI)

# ---------- HARI 1: data masuk dalam dua batch ----------
write_deltalake(PATH_2HARI, data_delta.iloc[:30_000], mode="overwrite")       # versi 0
write_deltalake(PATH_2HARI, data_delta.iloc[30_000:50_000], mode="append")    # versi 1
dt2 = DeltaTable(PATH_2HARI)
VERSI_AKHIR_HARI_1 = dt2.version()
WAKTU_AKHIR_HARI_1 = datetime.now(timezone.utc)
akhir_hari_1 = dt2.to_pandas()
print(f"AKHIR HARI 1 -> versi {VERSI_AKHIR_HARI_1} | baris {len(akhir_hari_1):,} | "
      f"total_bayar {akhir_hari_1['total_bayar'].sum():,.2f}")
time.sleep(2)   # jeda supaya stempel waktu hari 1 dan hari 2 berbeda

# ---------- HARI 2: tambah data baru, lalu koreksi beberapa baris ----------
write_deltalake(PATH_2HARI, data_delta.iloc[50_000:70_000], mode="append")    # versi 2
sekarang2 = data_delta.iloc[:70_000]
n_koreksi2 = int(((sekarang2["wilayah_jemput"] == "Brooklyn") & (sekarang2["metode_bayar"] == "Tunai")).sum())
DeltaTable(PATH_2HARI).update(predicate="wilayah_jemput = 'Brooklyn' AND metode_bayar = 'Tunai'",
                              updates={"total_bayar": "total_bayar + 0.5"})  # versi 3
akhir_hari_2 = DeltaTable(PATH_2HARI).to_pandas()
print(f"AKHIR HARI 2 -> versi {DeltaTable(PATH_2HARI).version()} | baris {len(akhir_hari_2):,} | "
      f"total_bayar {akhir_hari_2['total_bayar'].sum():,.2f} | baris dikoreksi {n_koreksi2:,}")

# ---------- Membaca tabel seperti keadaan AKHIR HARI 1 ----------
# (a) berdasarkan nomor versi
baca_v = DeltaTable(PATH_2HARI, version=VERSI_AKHIR_HARI_1).to_pandas()
print(f"\nTime travel versi {VERSI_AKHIR_HARI_1} -> baris {len(baca_v):,} | total_bayar {baca_v['total_bayar'].sum():,.2f}")
print("sama persis dengan akhir hari 1?",
      len(baca_v) == len(akhir_hari_1) and round(baca_v["total_bayar"].sum(), 2) == round(akhir_hari_1["total_bayar"].sum(), 2))

# (b) berdasarkan waktu (stempel waktu akhir hari 1)
try:
    dt_waktu = DeltaTable(PATH_2HARI)
    dt_waktu.load_as_version(WAKTU_AKHIR_HARI_1)
    baca_t = dt_waktu.to_pandas()
    print(f"Time travel waktu {WAKTU_AKHIR_HARI_1:%Y-%m-%d %H:%M:%S} UTC -> versi {dt_waktu.version()} | baris {len(baca_t):,}")
except Exception as e:
    print("Time travel berdasarkan waktu tidak didukung versi deltalake ini:", type(e).__name__)

print("\nRiwayat tabel:")
for h in DeltaTable(PATH_2HARI).history():
    print(" versi", h.get("version"), "-", h.get("operation"))

AKHIR HARI 1 -> versi 1 | baris 50,000 | total_bayar 1,366,574.00
AKHIR HARI 2 -> versi 3 | baris 70,000 | total_bayar 1,915,419.34 | baris dikoreksi 55

Time travel versi 1 -> baris 50,000 | total_bayar 1,366,574.00
sama persis dengan akhir hari 1? True
Time travel waktu 2026-10-29 12:18:59 UTC -> versi 1 | baris 50,000

Riwayat tabel:
 versi 3 - UPDATE
 versi 2 - WRITE
 versi 1 - WRITE
 versi 0 - WRITE


In [32]:
# TUGAS 3 (Sel 2) — Isi _delta_log/ (ringkas di layar + lengkap ke file .txt)
log_dir = f"{PATH_2HARI}/_delta_log"
isi_lengkap = []
for f in sorted(glob.glob(f"{log_dir}/*.json")):
    nama = os.path.basename(f)
    print(f"=== {nama} ===")
    with open(f) as fh:
        teks = fh.read()
    isi_lengkap.append(f"=== {nama} ===\n{teks}")
    for baris in teks.splitlines():
        aksi = json.loads(baris)
        jenis = next(iter(aksi))
        if jenis == "commitInfo":
            info = aksi[jenis]
            print(f"  commitInfo : operation={info.get('operation')}, timestamp={info.get('timestamp')}")
        elif jenis in ("add", "remove"):
            print(f"  {jenis:<10} : {aksi[jenis]['path']}")
        else:
            print(f"  {jenis}")

with open(f"{DIR_SIMPAN}/isi_delta_log_2hari.txt", "w") as fh:
    fh.write("\n\n".join(isi_lengkap))
print("\nIsi lengkap _delta_log/ tersimpan:", f"{DIR_SIMPAN}/isi_delta_log_2hari.txt")

=== 00000000000000000000.json ===
  commitInfo : operation=WRITE, timestamp=1793276339614
  protocol
  metaData
  add        : part-00000-562743b3-82ac-4a8c-ad6f-e18ab4be5486-c000.snappy.parquet
=== 00000000000000000001.json ===
  commitInfo : operation=WRITE, timestamp=1793276339642
  add        : part-00000-ad5cfe91-dcf1-42d7-ad16-9d5bcd374ef6-c000.snappy.parquet
=== 00000000000000000002.json ===
  commitInfo : operation=WRITE, timestamp=1793276341698
  add        : part-00000-5d61f8a6-9c4d-4a37-ad26-1a237f70895f-c000.snappy.parquet
=== 00000000000000000003.json ===
  commitInfo : operation=UPDATE, timestamp=1793276341768
  add        : part-00000-8d70d829-387a-486a-b902-eb43cf1421d7-c000.snappy.parquet
  remove     : part-00000-5d61f8a6-9c4d-4a37-ad26-1a237f70895f-c000.snappy.parquet
  remove     : part-00000-ad5cfe91-dcf1-42d7-ad16-9d5bcd374ef6-c000.snappy.parquet
  remove     : part-00000-562743b3-82ac-4a8c-ad6f-e18ab4be5486-c000.snappy.parquet

Isi lengkap _delta_log/ tersimpan: 

In [35]:
# ZIP folder trips_delta/ + daftar file pengumpulan
zip_path = shutil.make_archive(f"{DIR_SIMPAN}/trips_delta", "zip",
                               root_dir=os.path.dirname(PATH_DELTA), base_dir=os.path.basename(PATH_DELTA))
print("Tersimpan:", zip_path)

print("\nIsi folder Praktikum4:")
for f in sorted(os.listdir(DIR_SIMPAN)):
    print(" -", f)

Tersimpan: /content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip

Isi folder Praktikum4:
 - benchmark_nosql.csv
 - isi_delta_log_2hari.txt
 - tabel_rekap_k9.csv
 - trips_delta
 - trips_delta.zip
 - trips_delta_2hari
